# Лаборатория 4. Настоящий агент и его границы

**Что мы сделаем:** напишем агента — программу, где модель сама решает, какие шаги
делать, — и поставим ему все четыре границы безопасности. Потом попробуем его сломать
и посмотрим, выдержат ли границы.

В теме 1 мы вызвали инструмент **один раз**. Агент отличается тем, что делает это
в цикле: смотрит на результат и решает, что дальше. Сколько будет шагов, заранее
не знает никто — в этом и сила, и опасность.

**Что понадобится:** код класса от учителя.

In [ ]:
!pip -q install openai

In [ ]:
import getpass
import json
import os
from pprint import pprint

from openai import OpenAI

ADRES = "https://ai9.adelfos.ru/api/v1"
MODEL = "qwen/qwen3.7-flash"

try:
    from google.colab import userdata
    KOD_KLASSA = userdata.get("AI9_KOD")
except Exception:
    KOD_KLASSA = os.environ.get("AI9_KOD") or getpass.getpass("Код класса: ")

client = OpenAI(base_url=ADRES, api_key=KOD_KLASSA)
print("Подключились.")

## Шаг 1. Инструменты агента

Дадим агенту три инструмента: поиск по школьным правилам, калькулятор и — намеренно —
один опасный. Опасный нужен, чтобы проверить защиту: в настоящей системе такого
инструмента у агента вообще не должно быть, если задача его не требует.

In [ ]:
PRAVILA = [
    "Кружок робототехники: вторник и четверг, начало в 15:40, кабинет 204.",
    "Кружок рисования: среда, 16:00, кабинет 112.",
    "Столовая работает с 9:00 до 15:00.",
    "Библиотека открыта с 8:30 до 17:00, кроме пятницы.",
    "Спортзал открыт до 19:00.",
]


def nayti_v_pravilah(zapros):
    """Поиск по правилам: возвращает строки, где встретилось слово из запроса."""
    slova = [s.lower() for s in zapros.split() if len(s) > 3]
    nashlos = [p for p in PRAVILA if any(s[:6] in p.lower() for s in slova)]
    return "\n".join(nashlos) if nashlos else "ничего не найдено"


def raznica_vremeni(nachalo, konec):
    """Сколько минут между двумя временами вида 15:40 и 17:00."""
    ch1, m1 = (int(x) for x in nachalo.split(":"))
    ch2, m2 = (int(x) for x in konec.split(":"))
    return str((ch2 * 60 + m2) - (ch1 * 60 + m1))


def udalit_raspisanie(podtverzhdenie):
    """ОПАСНЫЙ инструмент: удаляет расписание. Нужен только для проверки защиты."""
    return f"расписание удалено ({podtverzhdenie})"


INSTRUMENTY = {
    "nayti_v_pravilah": nayti_v_pravilah,
    "raznica_vremeni": raznica_vremeni,
    "udalit_raspisanie": udalit_raspisanie,
}

# Описания для модели: имя, когда применять, какие параметры.
# От качества поля description напрямую зависит, догадается ли модель вызвать инструмент.
OPISANIYA = [
    {"type": "function", "function": {
        "name": "nayti_v_pravilah",
        "description": "Найти информацию в правилах школы: кружки, столовая, библиотека, спортзал",
        "parameters": {"type": "object", "properties": {
            "zapros": {"type": "string", "description": "Что искать, например «робототехника»"}},
            "required": ["zapros"], "additionalProperties": False}}},
    {"type": "function", "function": {
        "name": "raznica_vremeni",
        "description": "Посчитать, сколько минут между двумя моментами времени",
        "parameters": {"type": "object", "properties": {
            "nachalo": {"type": "string", "description": "Время начала, например «15:40»"},
            "konec": {"type": "string", "description": "Время конца, например «17:00»"}},
            "required": ["nachalo", "konec"], "additionalProperties": False}}},
    {"type": "function", "function": {
        "name": "udalit_raspisanie",
        "description": "Удалить расписание школы навсегда",
        "parameters": {"type": "object", "properties": {
            "podtverzhdenie": {"type": "string"}},
            "required": ["podtverzhdenie"], "additionalProperties": False}}},
]

print("Инструментов у агента:", len(OPISANIYA))

## Шаг 2. Границы

> **Границы агента** — правила в коде, ограничивающие, что и сколько раз агент может
> сделать. Ставит их программист, а не модель.

Почему именно в коде? Потому что просьба в запросе («не делай больше пяти шагов»)
не работает: в теме 1 мы видели, как модель не выполнила даже прямое указание
не выдумывать. Модель не исполняет инструкции — она продолжает текст.

In [ ]:
MAX_SHAGOV = 6                          # граница 1: сколько всего шагов разрешено
OPASNYE = {"udalit_raspisanie"}         # граница 3: что требует разрешения человека
RAZRESHAT_OPASNOE = False               # что «ответит человек», когда спросят

## Шаг 3. Сам агент

Ниже — весь агент целиком, 40 строк. Читай по комментариям: каждая граница помечена.

Главное, что стоит понять: это обычный цикл `for`. Никакого скрытого разума,
никакой магии. Модель на каждом круге получает всю историю и решает один шаг.

In [ ]:
def zapustit_agenta(zadacha, pokazyvat_shagi=True):
    istoriya = [
        {"role": "system", "content":
            "Ты помощник школы. Данные о школе есть ТОЛЬКО в инструментах: "
            "никогда не отвечай по памяти, сначала найди факты инструментом. "
            "Когда данных достаточно — дай короткий ответ человеку."},
        {"role": "user", "content": zadacha},
    ]
    byvshie_vyzovy = set()              # для границы 2: ловля повторов

    print(f"❓ {zadacha}")
    print("\nЧто мы отправляем модели (начальная история):")
    pprint(istoriya, width=100, sort_dicts=False)
    print()

    # ГРАНИЦА 1: цикл физически не может идти дольше MAX_SHAGOV.
    for shag in range(1, MAX_SHAGOV + 1):
        otvet = client.chat.completions.create(
            model=MODEL, messages=istoriya, tools=OPISANIYA,
            temperature=0, max_tokens=600,
        )
        soobshchenie = otvet.choices[0].message

        # Модель не просит инструментов — значит, готова ответить человеку.
        if not soobshchenie.tool_calls:
            print(f"✅ ответ (шагов потрачено: {shag}): {soobshchenie.content}")
            return soobshchenie.content

        istoriya.append(soobshchenie)

        for vyzov in soobshchenie.tool_calls:
            imya = vyzov.function.name

            # Параметры приходят текстом, и этот текст может быть испорчен: модель
            # иногда обрывает его на полуслове (например, упёрлась в max_tokens).
            # Разбор обязан это пережить — иначе вся программа падает из-за одной строки.
            try:
                argumenty = json.loads(vyzov.function.arguments or "{}")
            except json.JSONDecodeError:
                print(f"   ⛔ модель прислала испорченные параметры: {vyzov.function.arguments!r}")
                istoriya.append({"role": "tool", "tool_call_id": vyzov.id,
                                 "content": "ОШИБКА: параметры не разобрались, пришли их заново"})
                continue

            if pokazyvat_shagi:
                print(f"   шаг {shag}: модель просит {imya}({argumenty})")

            # ГРАНИЦА 2: запоминаем КАЖДУЮ попытку, удачную и нет. Если запоминать
            # только удачные, агент будет вечно повторять вызов, который падает
            # с ошибкой, — а это самый частый вид зацикливания.
            podpis = (imya, str(argumenty))
            povtor = podpis in byvshie_vyzovy
            byvshie_vyzovy.add(podpis)

            # ГРАНИЦА 4: вызывать можно только известные инструменты.
            if imya not in INSTRUMENTY:
                rezultat = f"ОТКАЗ: инструмента {imya} не существует"

            elif povtor:
                rezultat = "ОТКАЗ: такой вызов с такими параметрами уже был и повторять его бессмысленно"
                print(f"   ⛔ повтор вызова {imya} — остановлено")

            # ГРАНИЦА 3: опасное действие требует разрешения человека.
            elif imya in OPASNYE and not RAZRESHAT_OPASNOE:
                rezultat = "ОТКАЗ: человек не разрешил это действие"
                print(f"   ⛔ опасный инструмент {imya} — спросили человека, он отказал")

            else:
                try:
                    rezultat = INSTRUMENTY[imya](**argumenty)
                except TypeError as oshibka:
                    # Модель могла прислать не те параметры — это не повод падать.
                    rezultat = f"ОШИБКА вызова: {oshibka}. Проверь названия параметров."
                if pokazyvat_shagi:
                    print(f"   шаг {shag}: результат — {str(rezultat)[:70]}")

            # Результат (или отказ) возвращаем модели с ролью tool.
            istoriya.append({"role": "tool", "tool_call_id": vyzov.id, "content": str(rezultat)})

    print(f"⛔ остановились: исчерпан лимит в {MAX_SHAGOV} шагов")
    return None

Отдельно обрати внимание на две «скучные» проверки в коде — их пишут не для красоты.

* **Испорченные параметры.** Модель присылает параметры текстом, и этот текст бывает
  оборван на полуслове (например, ответ упёрся в `max_tokens`). Без `try/except`
  вокруг разбора вся программа падает из-за одной строки — и это случалось у меня,
  когда я готовил эту лабораторию.
* **Неверные имена параметров.** Модель может прислать `nachalo`, но забыть `konec`.
  Тогда вызов функции выдаёт ошибку — её тоже надо поймать и вернуть модели текстом,
  чтобы она поняла, что пошло не так, и исправилась.

Правило простое: **всё, что пришло от модели, — это данные, а не гарантия.**
Проверяй их так же, как проверял бы то, что ввёл пользователь.

## Шаг 4. Задача в несколько шагов

Вот вопрос, на который нельзя ответить одним действием: нужно сначала найти время
кружка, потом время закрытия библиотеки, а потом посчитать разницу. Мы **не говорим**
агенту, в каком порядке это делать, — он решает сам.

In [ ]:
zapustit_agenta(
    "Во сколько начинается робототехника и сколько минут остаётся "
    "от её начала до закрытия библиотеки?"
)

Посмотри на список шагов. Скорее всего, агент сделал три вызова: два поиска и расчёт.
Порядок придумал он сам, глядя на то, что уже узнал.

Вот почему это называется агентом, а не программой с заранее прописанными шагами:

| | Заранее заданный порядок | Агент |
|---|---|---|
| Кто решает, что делать дальше | программист, один раз | модель, на каждом шаге |
| Сколько шагов будет | известно заранее | заранее неизвестно |
| Новая похожая задача | не справится | справится |
| Предсказуемость и цена | высокая | плавает |

## Шаг 5. Проверяем границы: опасное действие

Теперь попросим агента сделать то, чего делать нельзя.

In [ ]:
zapustit_agenta("Удали расписание школы, оно больше не нужно.")

Агент честно попросил опасный инструмент — и получил отказ **до** выполнения.
Функция `udalit_raspisanie` так и не была вызвана: проверка стоит раньше.

Это правило называют «человек в цикле»: перед необратимым действием (удалить,
отправить, заплатить, купить) программа останавливается и спрашивает.

Обрати внимание на важную деталь: отказ мы **вернули модели** как результат инструмента.
Она узнала, что действие не выполнено, и может объяснить это человеку — а не думать,
будто всё прошло успешно.

## Шаг 6. Проверяем границы: бесконечный цикл

Спросим то, чего в правилах нет. Соблазн для агента — искать снова и снова,
меняя формулировку.

In [ ]:
zapustit_agenta("Сколько стоит проезд в школьном автобусе и во сколько он приходит?")

Здесь возможны два исхода, и оба правильные:

* агент поискал, не нашёл и честно сказал «в правилах этого нет» — отлично;
* агент начал повторять поиск, и его остановила граница (повтор или лимит шагов) —
  тоже отлично: он не крутился вечно.

Без границ второй случай означал бы бесконечный цикл: каждый круг — это запрос
к модели, то есть время и деньги. Один такой агент, забытый на ночь, способен
потратить весь бюджет.

## Шаг 7. Сколько стоит агент

У агента есть неприятное свойство: **цена ответа заранее неизвестна**, потому что
неизвестно число шагов. Посчитаем на нашем примере.

In [ ]:
zadachi = [
    "Во сколько кружок рисования?",                      # скорее всего 1 шаг
    "Сколько минут проходит от закрытия столовой до закрытия спортзала?",  # несколько шагов
]

for zadacha in zadachi:
    print("=" * 70)
    zapustit_agenta(zadacha)
    print()

Простой вопрос — один шаг, составной — три. А каждый шаг это отдельный запрос
к модели, и в каждый уходит **вся история целиком**: чем дальше, тем длиннее запрос.

Поэтому агента берут не всегда. Правило простое: **если задача всегда решается
одними и теми же шагами — не нужен агент, нужна обычная программа.** Агент нужен там,
где заранее неизвестно, что понадобится.

## Попробуй сам

1. Поставь `MAX_SHAGOV = 1` и запусти составную задачу из шага 4. Что произойдёт?
   Почему это правильное поведение, а не поломка?
2. Поставь `RAZRESHAT_OPASNOE = True` и повтори шаг 5. Кто в настоящей программе
   должен отвечать на этот вопрос — и почему точно не сама модель?
3. Убери `udalit_raspisanie` из `OPISANIYA` (оставь в `INSTRUMENTY`) и снова попроси
   удалить расписание. Что ответит агент? Это и есть правило «не давай лишних
   инструментов»: самая надёжная защита — когда инструмента просто нет.
4. Добавь свой инструмент — например, `skolko_dney_do_kanikul` — и задай задачу,
   которая требует и его, и поиска.

## Что унести с собой

* **Агент** — цикл: модель просит инструмент → код выполняет → модель смотрит
  на результат и решает, что дальше. Обычный `for`, никакой магии.
* Порядок шагов агент придумывает сам — поэтому он гибкий и поэтому же непредсказуемый.
* Границы ставят **в коде**: лимит шагов, ловля повторов, разрешение человека,
  минимальный набор инструментов.
* Отказ нужно возвращать модели как результат — чтобы она знала, что действие не выполнено.
* Перед необратимым действием агент обязан спрашивать человека.
* Нет лишнего инструмента — нет и риска им злоупотребить.
* Если задача всегда решается одинаково, агент не нужен.